<a href="https://colab.research.google.com/github/abilashkannanv/AIML/blob/main/Project%20MLOPS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import os

# Define the path to train.py
train_py_path = '/content/student_performance_project/app/train.py'

# Content for train.py
train_script_content = '''
import pandas as pd
from sklearn.linear_model import LinearRegression
import joblib
import os

# Define base directory relative to the script location
# This assumes the script is run from the project_root or app directory
project_root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))

# Paths to data and model directories
data_path = os.path.join(project_root, 'data', 'student_scores.csv')
model_save_path = os.path.join(project_root, 'model', 'model.pkl')

print(f"Loading data from: {data_path}")

# 1. Load student_scores.csv
try:
    df = pd.read_csv(data_path)
    print("Data loaded successfully.")
    # Separating print statements to avoid string literal issues with df.head()
    print("Data head:")
    print(df.head())
except FileNotFoundError:
    print(f"Error: student_scores.csv not found at {data_path}. Please ensure the file exists.")
    exit()

# 2. Use features: StudyHours, PreviousScore, SleepHours
features = ['StudyHours', 'PreviousScore', 'SleepHours']
target = 'PerformanceIndex'

X = df[features]
y = df[target]

print(f"Features used: {features}")
print(f"Target variable: {target}")

# 3. Train a Linear Regression model
print("Training Linear Regression model...")
model = LinearRegression()
model.fit(X, y)
print("Model training complete.")

# 4. Save the model to model/model.pkl using joblib
# Ensure the model directory exists
os.makedirs(os.path.dirname(model_save_path), exist_ok=True)
joblib.dump(model, model_save_path) # Corrected typo
print(f"Model saved successfully to: {model_save_path}")

print("Training pipeline finished.")
'''

# Write the content to train.py
with open(train_py_path, 'w') as f:
    f.write(train_script_content)

print(f"Content written to {train_py_path}")

Content written to /content/student_performance_project/app/train.py


In [16]:
# Change directory to the 'app' folder to execute the script relative to its location
%cd /content/student_performance_project/app

# Execute the train.py script
%run train.py

# Change back to the content directory if needed
%cd /content

# Verify the model file exists
model_file_path = '/content/student_performance_project/model/model.pkl'
if os.path.exists(model_file_path):
    print(f"\nSuccess: Model file found at {model_file_path}")
else:
    print(f"\nError: Model file NOT found at {model_file_path}")

/content/student_performance_project/app
Loading data from: /content/student_performance_project/data/student_scores.csv
Data loaded successfully.
Data head:
   StudyHours  PreviousScore  SleepHours  PerformanceIndex
0           2             60           7                55
1           3             65           6                62
2           5             70           8                70
3           4             75           7                78
4           6             80           6                85
Features used: ['StudyHours', 'PreviousScore', 'SleepHours']
Target variable: PerformanceIndex
Training Linear Regression model...
Model training complete.
Model saved successfully to: /content/student_performance_project/model/model.pkl
Training pipeline finished.
/content

Success: Model file found at /content/student_performance_project/model/model.pkl


Now, let's create the Gradio application in `app.py`. This application will:

1.  **Load the pre-trained model** (`model.pkl`).
2.  **Accept 3 numeric inputs** (`StudyHours`, `PreviousScore`, `SleepHours`).
3.  **Return a predicted `PerformanceIndex`**.
4.  **Log predictions** to `logs.txt` with timestamp, input values, and predicted score.
5.  Provide a **two-tab UI**:
    *   **Predict Score**: For making new predictions.
    *   **Monitoring Dashboard**: To display prediction statistics and alerts.

In [43]:
import os

# Define the path to app.py
app_py_path = '/content/student_performance_project/app/app.py'

# Content for app.py
app_script_content = '''
import gradio as gr
import joblib
import pandas as pd
import os
from datetime import datetime
import time
# New imports for Prometheus
from prometheus_client import generate_latest, Counter, Histogram
from starlette.responses import PlainTextResponse


# Define base directory relative to the script location
project_root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))

# Paths to model and logs
MODEL_PATH = os.path.join(project_root, 'model', 'model.pkl')
LOG_PATH = os.path.join(os.path.dirname(os.path.abspath(__file__)), 'logs.txt') # logs.txt is in the app directory

model = None
model_load_error = False
try:
    model = joblib.load(MODEL_PATH)
    print(f"Model loaded successfully from {MODEL_PATH}")
except FileNotFoundError:
    model_load_error = True
    print(f"Error: Model file not found at {MODEL_PATH}. Prediction functionality will be disabled.")
except Exception as e:
    model_load_error = True
    print(f"Error loading model from {MODEL_PATH}: {e}")

# Ensure logs.txt exists
if not os.path.exists(LOG_PATH):
    with open(LOG_PATH, 'w') as f:
        f.write("timestamp,study_hours,previous_score,sleep_hours,predicted_performance_index\\n")

# Prometheus Metrics
PREDICTION_REQUESTS = Counter(
    'student_performance_prediction_requests_total',
    'Total number of prediction requests.'
)
PREDICTION_LATENCY = Histogram(
    'student_performance_prediction_latency_seconds',
    'Latency of prediction requests in seconds.'
)
# LAST_PREDICTED_SCORE = Gauge('student_performance_last_predicted_score', 'Last predicted performance index.')


def log_prediction(timestamp, study_hours, previous_score, sleep_hours, predicted_score):
    with open(LOG_PATH, 'a') as f:
        f.write(f"{timestamp},{study_hours},{previous_score},{sleep_hours},{predicted_score:.2f}\\n")

def predict_score(study_hours, previous_score, sleep_hours):
    PREDICTION_REQUESTS.inc() # Increment counter for each request
    with PREDICTION_LATENCY.time(): # Measure latency
        if model_load_error or model is None:
            return "Error: Model not loaded. Cannot make predictions."

        try:
            # Create a DataFrame for prediction
            input_data = pd.DataFrame([{
                'StudyHours': study_hours,
                'PreviousScore': previous_score,
                'SleepHours': sleep_hours
            }])

            # Make prediction
            prediction = model.predict(input_data)[0]

            # Log the prediction
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            log_prediction(timestamp, study_hours, previous_score, sleep_hours, prediction)

            # LAST_PREDICTED_SCORE.set(prediction) # Set last predicted score if using Gauge

            return f"Predicted Performance Index: {prediction:.2f}"
        except Exception as e:
            return f"Prediction error: {e}"

def read_logs():
    if not os.path.exists(LOG_PATH):
        return []
    with open(LOG_PATH, 'r') as f:
        # Skip header if it exists
        lines = f.readlines()
        if lines and lines[0].startswith("timestamp,"):
            return [line.strip() for line in lines[1:]]
        return [line.strip() for line in lines]

def get_monitoring_data():
    logs = read_logs()
    total_predictions = len(logs)
    predicted_scores = []

    for entry in logs:
        try:
            parts = entry.split(',')
            if len(parts) == 5: # Expecting 5 parts: timestamp, SH, PS, SL, PPI
                predicted_scores.append(float(parts[4]))
        except (ValueError, IndexError):
            continue # Skip malformed log entries

    average_predicted_score = sum(predicted_scores) / total_predictions if total_predictions > 0 else 0
    last_10_log_entries = "\\n".join(logs[-10:]) if logs else "No predictions yet."

    alerts = []
    if model_load_error:
        alerts.append("**ALERT: Model not found or failed to load!**")
    if total_predictions > 0:
        if average_predicted_score > 90:
            alerts.append("**ALERT: Unusually high average predicted scores.**")
        elif average_predicted_score < 60:
            alerts.append("**ALERT: Unusually low average predicted scores.**")

    return (
        f"Total Predictions: {total_predictions}",
        f"Average Predicted Score: {average_predicted_score:.2f}",
        last_10_log_entries,
        "\\n".join(alerts) if alerts else "No active alerts." # Double-escaped \n
    )

# Gradio Interface

# Tab 1: Predict Score
predict_tab = gr.Interface(
    fn=predict_score,
    inputs=[
        gr.Number(label="Study Hours"),
        gr.Number(label="Previous Score"),
        gr.Number(label="Sleep Hours")
    ],
    outputs=gr.Textbox(label="Prediction Result"),
    title="Student Performance Predictor - Predict Score",
    description="Enter the student's study hours, previous score, and sleep hours to predict their performance index.",
    live=False
)

# Tab 2: Monitoring Dashboard - Using individual components within Blocks with a refresh button
with gr.Blocks() as dashboard_tab:
    gr.Markdown("# Monitoring Dashboard")
    gr.Markdown("--- Click Refresh to update --- ")

    total_predictions_output = gr.Textbox(label="Total Predictions", interactive=False)
    avg_predicted_score_output = gr.Textbox(label="Average Predicted Score", interactive=False)
    last_10_logs_output = gr.Textbox(label="Last 10 Log Entries", interactive=False, lines=10)
    alerts_output = gr.Markdown("## Alerts")

    refresh_btn = gr.Button("Refresh Dashboard")

    # Initial load of data
    initial_total, initial_avg, initial_logs, initial_alerts = get_monitoring_data()
    total_predictions_output.value = initial_total
    avg_predicted_score_output.value = initial_avg
    last_10_logs_output.value = initial_logs
    alerts_output.value = initial_alerts

    # Define how components update on button click
    refresh_btn.click(
        fn=get_monitoring_data,
        inputs=[],
        outputs=[
            total_predictions_output,
            avg_predicted_score_output,
            last_10_logs_output,
            alerts_output
        ]
    )


# Combine tabs
app = gr.TabbedInterface([
    predict_tab,
    dashboard_tab # Use the Blocks object directly
], ["Predict Score", "Monitoring Dashboard"])

# Add Prometheus /metrics endpoint to the Gradio FastAPI app
@app.app.get("/metrics") # Access the underlying FastAPI app instance
async def metrics():
    return PlainTextResponse(generate_latest().decode("utf-8"))

if __name__ == "__main__":
    app.launch(debug=True, share=True)
'''

# Write the content to app.py
with open(app_py_path, 'w') as f:
    f.write(app_script_content)

print(f"Content written to {app_py_path}")

Content written to /content/student_performance_project/app/app.py


Now, let's run the `app.py` script to launch the Gradio application.

In [ ]:
# Change directory to the 'app' folder to execute the script relative to its location
%cd /content/student_performance_project/app

# Install/Upgrade necessary libraries, including gradio, to ensure compatibility
!pip install gradio joblib pandas scikit-learn --upgrade -q

# Execute the app.py script
# The app.launch(share=True) will provide a public URL.
# Keep this cell running to keep the Gradio app alive.
!python app.py

/content/student_performance_project/app
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LinearRegression from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
Model loaded successfully from /content/student_performance_project/model/model.pkl
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://3079c9b147a2919f3c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [6]:
import pandas as pd
import os

# Define the path for the data file
data_dir = '/content/student_performance_project/data'
student_scores_csv_path = os.path.join(data_dir, 'student_scores.csv')

# Create some dummy data
data = {
    'StudyHours': [2, 3, 5, 4, 6, 7, 8, 5, 9, 10],
    'PreviousScore': [60, 65, 70, 75, 80, 85, 90, 72, 95, 98],
    'SleepHours': [7, 6, 8, 7, 6, 8, 7, 7, 9, 8],
    'PerformanceIndex': [55, 62, 70, 78, 85, 90, 92, 75, 98, 100]
}
df_sample = pd.DataFrame(data)

# Save the dummy data to student_scores.csv
df_sample.to_csv(student_scores_csv_path, index=False)

print(f"Dummy data saved to: {student_scores_csv_path}")
display(df_sample.head())

Dummy data saved to: /content/student_performance_project/data/student_scores.csv


,StudyHours,PreviousScore,SleepHours,PerformanceIndex
0,2,60,7,55
1,3,65,6,62
2,5,70,8,70
3,4,75,7,78
4,6,80,6,85


In [41]:
# Change directory to the 'app' folder to execute the script relative to its location
%cd /content/student_performance_project/app

# Install/Upgrade necessary libraries, including gradio, to ensure compatibility
!pip install gradio joblib pandas scikit-learn --upgrade -q

# Execute the app.py script
# The app.launch(share=True) will provide a public URL.
# Keep this cell running to keep the Gradio app alive.
!python app.py

/content/student_performance_project/app
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LinearRegression from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
Model loaded successfully from /content/student_performance_project/model/model.pkl
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://7d28d3012fe003607d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Keyboard interruption in main thread... closing server.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", li

In [40]:
import os

app_py_path = '/content/student_performance_project/app/app.py'

print(f"Reading content from: {app_py_path}")

if os.path.exists(app_py_path):
    with open(app_py_path, 'r') as f:
        read_content = f.read()
    print("--- Current app.py Content ---")
    print(read_content)
    print("------------------------------")
else:
    print(f"Error: {app_py_path} not found.")

Reading content from: /content/student_performance_project/app/app.py
--- Current app.py Content ---

import gradio as gr
import joblib
import pandas as pd
import os
from datetime import datetime
import time

# Define base directory relative to the script location
project_root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))

# Paths to model and logs
MODEL_PATH = os.path.join(project_root, 'model', 'model.pkl')
LOG_PATH = os.path.join(os.path.dirname(os.path.abspath(__file__)), 'logs.txt') # logs.txt is in the app directory

model = None
model_load_error = False
try:
    model = joblib.load(MODEL_PATH)
    print(f"Model loaded successfully from {MODEL_PATH}")
except FileNotFoundError:
    model_load_error = True
    print(f"Error: Model file not found at {MODEL_PATH}. Prediction functionality will be disabled.")
except Exception as e:
    model_load_error = True
    print(f"Error loading model from {MODEL_PATH}: {e}")

# Ensure logs.txt exists
if not os.path.exists(LOG_PA

### Update `requirements.txt`

To ensure all necessary libraries are installed with specific versions, let's update the `requirements.txt` file.

In [36]:
import os

project_root = '/content/student_performance_project'
requirements_path = os.path.join(project_root, 'requirements.txt')

requirements_content = '''
gradio==4.38.1
pandas==2.0.3
scikit-learn==1.3.0
joblib==1.3.2
'''

with open(requirements_path, 'w') as f:
    f.write(requirements_content.strip())

print(f"Content written to {requirements_path}")

Content written to /content/student_performance_project/requirements.txt


### Create `README.md`

Now, let's create a `README.md` file to document the project. This file will include instructions on how to run the training and application, explain the monitoring features, provide Docker build/run instructions, and list any assumptions or limitations.

In [37]:
import os

project_root = '/content/student_performance_project'
readme_path = os.path.join(project_root, 'README.md')

readme_content = '''
# Student Performance Prediction Project

This project implements a machine learning pipeline to predict student performance based on various factors, along with a Gradio-based application for predictions and monitoring.

## Project Structure

```
student_performance_project/
├── data/
│   └── student_scores.csv
├── model/
│   └── model.pkl
├── app/
│   ├── train.py
│   ├── app.py
│   ├── logs.txt
│   └── requirements.txt
└── README.md
```

## How to Run Training and Application

### 1. Data Preparation

Ensure `student_scores.csv` is present in the `data/` directory. If not, the notebook should contain cells to generate this data.

### 2. Train the Model

The `train.py` script trains a Linear Regression model and saves it to `model/model.pkl`.

To run the training:

```bash
# Navigate to the app directory
cd /content/student_performance_project/app
# Run the training script
python train.py
```

Alternatively, you can run the corresponding cells in the Jupyter/Colab notebook.

### 3. Launch the Gradio Application

The `app.py` script launches a Gradio web application for making predictions and monitoring.

To run the application:

```bash
# Navigate to the app directory
cd /content/student_performance_project/app
# Install necessary packages (if not already installed or if versions need updating)
pip install -r requirements.txt
# Run the application script
python app.py
```

When launched, Gradio will provide a local and a public URL. Access the public URL to interact with the application.

## How Monitoring Works

The Gradio application includes a "Monitoring Dashboard" tab. This dashboard provides:

*   **Total Predictions:** The total number of predictions made since the `logs.txt` file was created.
*   **Average Predicted Score:** The average performance index predicted across all logged predictions.
*   **Last 10 Log Entries:** Displays the most recent 10 prediction logs, including timestamp, input features, and predicted score.
*   **Alerts:** Notifies if the model failed to load or if the average predicted score falls outside an expected range (e.g., unusually high > 90 or low < 60).

The dashboard does not auto-refresh. To get updated monitoring data, you need to click the "Refresh Dashboard" button.

All predictions are logged to `app/logs.txt`.

## Docker Build/Run Instructions

To containerize this application using Docker:

### 1. Build the Docker Image

Navigate to the `student_performance_project/app` directory (where `Dockerfile` and `requirements.txt` are located) and run:

```bash
docker build -t student-performance-app .
```

### 2. Run the Docker Container

Run the built image, mapping a port (e.g., 7860) to access the Gradio application:

```bash
docker run -p 7860:7860 student-performance-app
```

Once the container is running, access the Gradio app via `http://localhost:7860` in your web browser.

## Assumptions and Limitations

*   **Model Type:** The project currently uses a simple Linear Regression model. For real-world scenarios, more complex models might be required.
*   **Data Availability:** Assumes `student_scores.csv` is available in the `data/` directory for training. The provided notebook generates dummy data for demonstration.
*   **Log Management:** Prediction logs are stored in a simple `logs.txt` file. For production, a more robust logging solution (e.g., a database, dedicated logging service) would be necessary.
*   **Monitoring Granularity:** The monitoring dashboard provides basic statistics. Advanced monitoring would include time-series analysis, drift detection, and more sophisticated alerting.
*   **Gradio `every` Parameter:** Due to limitations with `gr.Blocks` and `gr.Interface` within the Colab environment, the auto-refresh (`every`) functionality for the monitoring dashboard has been replaced with a manual refresh button.
*   **Feature Engineering:** No complex feature engineering is performed; features are used directly as provided.
*   **Version Pinning:** `requirements.txt` includes version pinning to ensure reproducibility, but updates to these versions might be necessary for newer environments or features.
'''

with open(readme_path, 'w') as f:
    f.write(readme_content)

print(f"Content written to {readme_path}")

Content written to /content/student_performance_project/README.md


### Create `Dockerfile`

Now, let's create a `Dockerfile` in the `app` directory. This Dockerfile will define the environment and steps needed to build a Docker image for our application, ensuring consistent execution across different environments.

In [38]:
import os

project_root = '/content/student_performance_project'
app_dir = os.path.join(project_root, 'app')
dockerfile_path = os.path.join(app_dir, 'Dockerfile')

dockerfile_content = '''
# Use an official Python runtime as a parent image
FROM python:3.10-slim

# Set the working directory in the container
WORKDIR /app

# Copy the entire project into the container
# First copy only the app directory content and requirements.txt
# Then copy the data and model directories outside of app
COPY ./app/requirements.txt /app/

# Install any needed packages specified in requirements.txt
RUN pip install --no-cache-dir -r requirements.txt

# Copy the rest of the app directory content
COPY ./app/ /app/

# Copy the data and model directories to the project root in the container
COPY ./data/ /app/data/
COPY ./model/ /app/model/

# Expose the port that Gradio uses
EXPOSE 7860

# Run the app.py script when the container launches
CMD ["python", "app.py"]
'''

with open(dockerfile_path, 'w') as f:
    f.write(dockerfile_content)

print(f"Content written to {dockerfile_path}")

Content written to /content/student_performance_project/app/Dockerfile


### Sample Prometheus Configuration (`prometheus.yml`)

This configuration file tells Prometheus where to find the metrics from your Gradio application. You'll need to create this file locally (outside of Colab) if you're setting up a Prometheus server.

In [44]:
prometheus_yml_content = '''
# my global config
global:
  scrape_interval:     15s # Set the scrape interval to every 15 seconds. Default is every 1 minute.
  evaluation_interval: 15s # Evaluate rules every 15 seconds. Default is every 1 minute.
  # scrape_timeout is set to the global default (10s).

# A scrape configuration containing exactly one endpoint to scrape:
# Here it's Prometheus itself.
scrape_configs:
  # The job name is added as a label `job=<job_name>` to any timeseries scraped from this config.
  - job_name: 'gradio_app'

    # metrics_path defaults to '/metrics'
    # scheme defaults to 'http'.

    static_configs:
      - targets: ['[YOUR_GRADIO_PUBLIC_URL]'] # REPLACE WITH YOUR GRADIO PUBLIC URL (e.g., 'https://<your-id>.gradio.live')
        scheme: https # Use https if your Gradio public URL is https
'''

print(prometheus_yml_content)


# my global config
global:
  scrape_interval:     15s # Set the scrape interval to every 15 seconds. Default is every 1 minute.
  evaluation_interval: 15s # Evaluate rules every 15 seconds. Default is every 1 minute.
  # scrape_timeout is set to the global default (10s).

# A scrape configuration containing exactly one endpoint to scrape:
# Here it's Prometheus itself.
scrape_configs:
  # The job name is added as a label `job=<job_name>` to any timeseries scraped from this config.
  - job_name: 'gradio_app'

    # metrics_path defaults to '/metrics'
    # scheme defaults to 'http'.

    static_configs:
      - targets: ['[YOUR_GRADIO_PUBLIC_URL]'] # REPLACE WITH YOUR GRADIO PUBLIC URL (e.g., 'https://<your-id>.gradio.live')
        scheme: https # Use https if your Gradio public URL is https



### Instructions for Local Prometheus/Grafana Setup (Outside Colab)

To use this `prometheus.yml` and run Prometheus/Grafana locally to monitor your Gradio app:

1.  **Save the `prometheus.yml` content** to a file named `prometheus.yml` in a directory on your local machine (e.g., `~/monitoring`).
2.  **Replace `[YOUR_GRADIO_PUBLIC_URL]`** in `prometheus.yml` with the actual public URL printed by your Gradio app (e.g., `https://<your-id>.gradio.live`).
3.  **Create a `docker-compose.yml`** file in the *same local directory* as `prometheus.yml` with the following content:

    ```yaml
    version: '3.8'

    services:
      prometheus:
        image: prom/prometheus
        container_name: prometheus
        ports:
          - "9090:9090"
        volumes:
          - ./prometheus.yml:/etc/prometheus/prometheus.yml
        command:
          - '--config.file=/etc/prometheus/prometheus.yml'
          - '--web.enable-remote-write-receiver'
        restart: unless-stopped

      grafana:
        image: grafana/grafana
        container_name: grafana
        ports:
          - "3000:3000"
        volumes:
          - grafana-data:/var/lib/grafana
        restart: unless-stopped

    volumes:
      grafana-data:
    ```

4.  **Run Docker Compose**: Open a terminal in that local directory and run:

    ```bash
    docker compose up -d
    ```

5.  **Access Prometheus**: Go to `http://localhost:9090` in your local browser to see the Prometheus UI. You should see `gradio_app` as a target.

6.  **Access Grafana**: Go to `http://localhost:3000` (default login: `admin`/`admin`). Add Prometheus as a data source (Configuration -> Data Sources -> Add data source -> Prometheus) pointing to `http://prometheus:9090` (because Grafana is in the same Docker network as Prometheus).

7.  **Create a Grafana Dashboard**: You can now create a new dashboard and add panels to visualize metrics like `student_performance_prediction_requests_total` or `student_performance_prediction_latency_seconds_sum / student_performance_prediction_latency_seconds_count` (for average latency).

### How to Create a Grafana Dashboard for Gradio Metrics

These steps assume you have Prometheus and Grafana running locally via `docker-compose` and that Prometheus is configured to scrape your Gradio app's public URL.

#### Step 1: Access Grafana

1.  Open your web browser and navigate to `http://localhost:3000`.
2.  Log in using the default credentials: `Username: admin`, `Password: admin`.
3.  You will be prompted to change the password. You can do this or skip it for now.

#### Step 2: Add Prometheus as a Data Source

1.  In the Grafana UI, click on the **Gear icon** (Configuration) in the left-hand navigation bar.
2.  Select **Data Sources**.
3.  Click on **Add data source**.
4.  Search for and select **Prometheus**.
5.  Configure the Prometheus data source:
    *   **Name:** `Prometheus-Gradio` (or any name you prefer)
    *   **URL:** `http://prometheus:9090` (This URL works because Grafana and Prometheus are running in the same Docker Compose network and can resolve each other by their service names).
    *   Leave other settings as default for now.
6.  Click **Save & test**. You should see a message like "Data source is working".

#### Step 3: Create a New Dashboard

1.  Click on the **Plus icon** (Create) in the left-hand navigation bar.
2.  Select **Dashboard**.
3.  Click on **Add new panel**.

#### Step 4: Add Panels for Your Metrics

Let's add two panels: one for total prediction requests and another for average prediction latency.

**Panel 1: Total Prediction Requests**

1.  In the new panel editor, under the **Query** tab, ensure your `Prometheus-Gradio` data source is selected.
2.  In the **Metric browser** or **PromQL** field, enter the following query:
    ```promql
    student_performance_prediction_requests_total
    ```
3.  In the **Legend** field (usually below the query), you can change it to `Prediction Requests` for better readability.
4.  On the right-hand **Panel options** tab:
    *   Set **Title:** `Total Prediction Requests`
    *   Set **Visualization:** `Graph` (or `Stat` if you want just a single number)
5.  Click **Apply** to add the panel to your dashboard.

**Panel 2: Average Prediction Latency**

1.  Click **Add panel** > **Add new panel** again.
2.  In the **Query** tab, enter the following PromQL query to calculate the average latency:
    ```promql
    rate(student_performance_prediction_latency_seconds_sum[5m]) / rate(student_performance_prediction_latency_seconds_count[5m])
    ```
    *   *Explanation:* This query calculates the average latency over the last 5 minutes by dividing the sum of latencies by the count of requests within that period.
3.  In the **Legend** field, you can set it to `Average Latency`.
4.  On the right-hand **Panel options** tab:
    *   Set **Title:** `Average Prediction Latency`
    *   Set **Visualization:** `Graph`
    *   Under **Standard options** > **Unit**, you might want to select `seconds` to display the values appropriately.
5.  Click **Apply**.

#### Step 5: Save Your Dashboard

1.  Click the **Save icon** at the top of the dashboard.
2.  Give your dashboard a **name** (e.g., `Gradio App Monitoring`).
3.  Click **Save**.

You now have a basic Grafana dashboard visualizing the metrics from your Gradio application! You can continue to add more panels, customize their appearance, and set up alerts within Grafana.

### Instructions for Local Prometheus/Grafana Setup (Outside Colab)

To use the `prometheus.yml` content and run Prometheus/Grafana locally to monitor your Gradio app:

1.  **Save the `prometheus.yml` content** to a file named `prometheus.yml` in a directory on your local machine (e.g., `~/monitoring`).
2.  **Replace `[YOUR_GRADIO_PUBLIC_URL]`** in `prometheus.yml` with the actual public URL printed by your Gradio app (e.g., `https://<your-id>.gradio.live`).
3.  **Create a `docker-compose.yml`** file in the *same local directory* as `prometheus.yml` with the following content:

    ```yaml
    version: '3.8'

    services:
      prometheus:
        image: prom/prometheus
        container_name: prometheus
        ports:
          - "9090:9090"
        volumes:
          - ./prometheus.yml:/etc/prometheus/prometheus.yml
        command:
          - '--config.file=/etc/prometheus/prometheus.yml'
          - '--web.enable-remote-write-receiver'
        restart: unless-stopped

      grafana:
        image: grafana/grafana
        container_name: grafana
        ports:
          - "3000:3000"
        volumes:
          - grafana-data:/var/lib/grafana
        restart: unless-stopped

    volumes:
      grafana-data:
    ```

4.  **Run Docker Compose**: Open a terminal in that local directory and run:

    ```bash
    docker compose up -d
    ```

5.  **Access Prometheus**: Go to `http://localhost:9090` in your local browser to see the Prometheus UI. You should see `gradio_app` as a target.

6.  **Access Grafana**: Go to `http://localhost:3000` (default login: `admin`/`admin`). Add Prometheus as a data source (Configuration -> Data Sources -> Add data source -> Prometheus) pointing to `http://prometheus:9090` (because Grafana is in the same Docker network as Prometheus).

7.  **Create a Grafana Dashboard**: You can now create a new dashboard and add panels to visualize metrics like `student_performance_prediction_requests_total` or `student_performance_prediction_latency_seconds_sum / student_performance_prediction_latency_seconds_count` (for average latency).

### How to Verify the Prometheus Metrics Endpoint

1.  **Launch Your Gradio App:** Make sure your Gradio application is running. You would typically do this by executing a cell like `!python app.py` (e.g., cell `5884de83`). When it launches, it will print a public URL.

    ```text
    * Running on public URL: https://<YOUR_UNIQUE_ID>.gradio.live
    ```

2.  **Identify the Public URL:** From the output of the Gradio app launch, copy the `public URL`. For example, `https://7d28d3012fe003607d.gradio.live` from a previous output.

3.  **Access the Metrics Endpoint:** Open a new tab in your web browser and append `/metrics` to the public URL. So, if your public URL is `https://<YOUR_UNIQUE_ID>.gradio.live`, you would navigate to:

    ```
    https://<YOUR_UNIQUE_ID>.gradio.live/metrics
    ```

4.  **Observe the Output:** If the endpoint is active and correctly configured, your browser should display a plain text page containing various Prometheus metrics. You should see metrics prefixed with `student_performance_` (e.g., `student_performance_prediction_requests_total`, `student_performance_prediction_latency_seconds`).

If you see this output, your Gradio app is successfully exposing Prometheus metrics!

First, let's create the `student_scores.csv` file with some dummy data, as the `train.py` script expects this file to exist.